# Idiom-Aware Fine-Tuning of EN-DE Translations

Idea: 2-stage fine-tuning of a pre-trained model to increase idiom translation accuracy without sacrificing general language translation quality

Specifically in this notebook I want to focus on Parameter Efficient Fine-tuning (PEFT) techniques.

---
**Stages**

---



**Stage 0**: Baseline (Helsikin-NLP model)

**Stage 1**: Immediate Idiom-only fine-tuning
- Increase the model's idiom translation performance by fine-tuning the baseline from stage 0 on EN-DE idiom sentence pairs (`IdiomsInCtx-MT`)
- This will likely increase idiom-specific translation performance (measured as `Idiom BLEU`), and likely decrease general language translation (measured as `WMT BLEU)` performance.

**Stage 2**: General Language fine-tuning
- Fine-tune the model from stage 1 on a general language EN-DE dataset (`WMT-14`)
- The hope is to maintain the learned idiom-translation ability from stage 1, while recovering some of the lost general language translation perfromance.

Ideally, our stage 2 model has not sacrificied its general language translation capabilties, but has notably increased idiom-specific translation capabilities.

---
**Experiments** in this notebook

---
- **Stage 1:** Idioms fine-tuning (LoRA)
- **Stage 2:** WMT fine-tuning (LoRA) with frozen encoder

With ranked decoding sweep {4,8,16} with
- greedy
- beam search with num_beams = {4,8}
- length_penalty = {0.8, 1.0, 1.2)
- early_stopping

## Clone Repo from Git / Commit Changes
Run the git clone everytime a new session is started

In [ ]:
!rm -rf Idiom-Aware-Finetuning-in-MT-EN-to-DE
!git clone https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
%cd Idiom-Aware-Finetuning-in-MT-EN-to-DE
!ls

Cloning into 'Idiom-Aware-Finetuning-in-MT-EN-to-DE'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 82 (delta 33), reused 30 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 188.04 KiB | 20.89 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/Idiom-Aware-Finetuning-in-MT-EN-to-DE


Copy notebook from Drive into Repo

In [ ]:
# !cp "/content/drive/MyDrive/ds266_idiom_mt/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb" notebooks/

Add and commit changes

In [ ]:
#!git config --global user.email "auntiepersilla@gmail.com"
#!git config --global user.name "Michael Strommer"

#!git add notebooks/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb
#!git commit -m "Add all-in-one Drive-first pipeline notebook"

[main 45cc75b] Add all-in-one Drive-first pipeline notebook
 1 file changed, 1 insertion(+)
 create mode 100644 notebooks/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb


Push to GitHub

In [ ]:
#!git remote set-url origin https://auntiepersilla:<TOKEN>@github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
#!git push origin main

Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 12 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 33.19 KiB | 5.53 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
   1e3ab78..45cc75b  main -> main


## Mount Drive, Create Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
CACHE_DIR = os.path.join(DRIVE_ROOT, "cache")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR

print("DRIVE_ROOT:", DRIVE_ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("CACHE_DIR:", CACHE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT: /content/drive/MyDrive/ds266_idiom_mt
CHECKPOINT_DIR: /content/drive/MyDrive/ds266_idiom_mt/checkpoints
RESULTS_DIR: /content/drive/MyDrive/ds266_idiom_mt/results
CACHE_DIR: /content/drive/MyDrive/ds266_idiom_mt/cache


## Install Dependencies & Imports

In [ ]:
!pip -q install -U "transformers>=4.38" datasets evaluate sacrebleu accelerate sentencepiece sacremoses "protobuf<6"

In [ ]:
import random, numpy as np, torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
import sacrebleu
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [ ]:
!pip -q install peft

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch.nn as nn

## Load Data

- Idiom-only Data: davidstap/IdiomsInCtx-MT (1000 EN-DE idiom sentence pairs)
  - train: 800
  - dev: 100
  - test: 100
- General Translation Training and Test Data: WMT-14 EN-DE
  - train: 20.000
  - test: 2.000
  - test: 3.003

In [ ]:
def load_idioms(name="davidstap/IdiomsInCtx-MT", config="en-de", train_frac=0.8, dev_frac=0.1, seed=42):
    raw = load_dataset(name, config)

    full = raw["test"] if ("test" in raw and len(raw.keys()) == 1) else raw[list(raw.keys())[0]]
    src_lang, tgt_lang = config.split("-")

    def standardize(ex):
        if "translation" in ex and src_lang in ex["translation"] and tgt_lang in ex["translation"]:
            return {"src": ex["translation"][src_lang], "tgt": ex["translation"][tgt_lang]}
        if src_lang in ex and tgt_lang in ex:
            return {"src": ex[src_lang], "tgt": ex[tgt_lang]}
        raise ValueError(list(ex.keys()))

    full = full.map(standardize)
    full = full.remove_columns([c for c in full.column_names if c not in ["src","tgt"]])

    tmp = full.train_test_split(test_size=(1-train_frac), seed=seed)
    train, rest = tmp["train"], tmp["test"]

    dev_frac_of_rest = dev_frac / (1-train_frac)
    tmp2 = rest.train_test_split(test_size=(1-dev_frac_of_rest), seed=seed)
    dev, test = tmp2["train"], tmp2["test"]

    return DatasetDict({"train": train, "dev": dev, "test": test})

def load_wmt14(ft_train_n=20000, ft_dev_n=2000, seed=42):
    wmt = load_dataset("wmt14", "de-en")
    def to_en_de(ex):
        tr = ex["translation"]
        return {"src": tr["en"], "tgt": tr["de"]}

    train = wmt["train"].map(to_en_de, remove_columns=wmt["train"].column_names)
    test = wmt["test"].map(to_en_de, remove_columns=wmt["test"].column_names)

    shuf = train.shuffle(seed=seed)
    ft_train = shuf.select(range(min(ft_train_n, len(shuf))))
    ft_dev = shuf.select(range(min(ft_train_n, len(shuf)), min(ft_train_n+ft_dev_n, len(shuf))))
    return DatasetDict({"ft_train": ft_train, "ft_dev": ft_dev, "wmt_test": test})

idiom_ds = load_idioms(seed=SEED)
general_ds = load_wmt14(seed=SEED)

print("idiom_ds:", {k: len(v) for k,v in idiom_ds.items()})
print("general_ds:", {k: len(v) for k,v in general_ds.items()})
print("idiom sample:", idiom_ds["test"][0])

idiom_ds: {'train': 800, 'dev': 100, 'test': 100}
general_ds: {'ft_train': 20000, 'ft_dev': 2000, 'wmt_test': 3003}
idiom sample: {'src': 'Not a word of all this to anyone!', 'tgt': 'Kein Wort von all dem zu irgendjemandem!'}


## Metrics and Generation Helpers

In [ ]:
# Set Global Hyperparameters for Predictions
max_source_len = 256
max_new_tokens = 128
batch_size = 16

In [ ]:
@torch.no_grad()
# Generate Predictions
def generate_preds(model, tok, sources, max_source_len=max_source_len, max_new_tokens=max_new_tokens, batch_size=batch_size, gen_kwargs=None):
    model.eval()
    gen_kwargs = gen_kwargs or {}
    preds=[]
    for i in range(0, len(sources), batch_size):
        batch = sources[i:i+batch_size]
        inputs = tok(batch, return_tensors="pt", truncation=True, padding=True, max_length=max_source_len).to(device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

DECODING_SETUPS = {
    "greedy": {},
    "beam4": {"num_beams": 4, "early_stopping": True},
    "beam8": {"num_beams": 8, "early_stopping": True, "length_penalty": 1.0},
}

# Compute Metrics
def compute_metrics(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"bleu": float(bleu), "chrf": float(chrf)}

In [ ]:
# Metrics Logger

METRICS_PATH = os.path.join(RESULTS_DIR, "metrics.csv")

METRIC_FIELDS = [
    "timestamp",
    "run_name",
    "split",
    "bleu",
    "chrf",
    "model_name",
    "learning_rate",
    "epochs",
    "batch_size",
    "max_source_len",
    "max_new_tokens",
    "freeze_encoder",
    "train_size",
    "dev_size",
    "seed",
    "n_eval",
    "notes",
]

def log_metrics(
    run_name,
    split,
    metrics,
    *,
    model_name=None,
    learning_rate=None,
    epochs=None,
    batch_size=None,
    max_source_len=None,
    max_new_tokens=None,
    freeze_encoder=None,
    train_size=None,
    dev_size=None,
    seed=None,
    n_eval=None,
    notes=None,
):
    # Initialize full schema with None
    row = {k: None for k in METRIC_FIELDS}

    row.update({
        "timestamp": datetime.now(ZoneInfo('America/Los_Angeles')).isoformat(timespec="seconds"),
        "run_name": run_name,
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "model_name": model_name,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "batch_size": batch_size,
        "max_source_len": max_source_len,
        "max_new_tokens": max_new_tokens,
        "freeze_encoder": freeze_encoder,
        "train_size": train_size,
        "dev_size": dev_size,
        "seed": seed,
        "n_eval": n_eval,
        "notes": notes,
    })

    df = pd.DataFrame([row])

    if os.path.exists(METRICS_PATH):
        df.to_csv(METRICS_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(METRICS_PATH, index=False)

    return row

# Stage 0: Baseline Evaluation
- Model: Helsinki-NLP/opus-mt-en-de

In [ ]:
# Load Model and Tokenizer
BASE_MODEL = "Helsinki-NLP/opus-mt-en-de"
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
#model.config

In [ ]:
# Set test data
idiom_test = idiom_ds["test"]
wmt_test = general_ds["wmt_test"]

# Global Prediction Hyperparameters
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128
BATCH_SIZE = 16

## Baseline Evaluation

In [ ]:
# Evaluate Idiom Translation
idiom_pred = generate_preds(model, tok, idiom_test["src"],
                            max_source_len=MAX_SOURCE_LEN,
                            max_new_tokens=MAX_NEW_TOKENS,
                            batch_size=BATCH_SIZE)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("baseline idioms:", idiom_m)

log_metrics(
    "baseline", "idioms_test", idiom_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(idiom_ds["train"]),
    dev_size=len(idiom_ds["dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes="pretrained baseline",
)

# Evaluate General Translation
wmt_pred = generate_preds(model, tok, wmt_test["src"],
                          max_source_len=MAX_SOURCE_LEN,
                          max_new_tokens=MAX_NEW_TOKENS,
                          batch_size=BATCH_SIZE)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("baseline wmt:", wmt_m)

log_metrics(
    "baseline", "wmt_test", wmt_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes="pretrained baseline (WMT test full 3003)",
)


baseline idioms: {'bleu': 39.64935194574556, 'chrf': 60.786111999956105}
baseline wmt: {'bleu': 27.567994028871393, 'chrf': 58.43184557779851}


{'timestamp': '2026-02-25T21:28:09-08:00',
 'run_name': 'baseline',
 'split': 'wmt_test',
 'bleu': 27.567994028871393,
 'chrf': 58.43184557779851,
 'model_name': 'Helsinki-NLP/opus-mt-en-de',
 'learning_rate': None,
 'epochs': None,
 'batch_size': 16,
 'max_source_len': 256,
 'max_new_tokens': 128,
 'freeze_encoder': False,
 'train_size': 20000,
 'dev_size': 2000,
 'seed': 42,
 'n_eval': 3003,
 'notes': 'pretrained baseline (WMT test full 3003)'}

## Stage 1: Idiom Fine-tuning

In [ ]:
def fine_tune(model_name_or_path, train_ds, dev_ds, out_dir, lr, epochs, batch_size=8, freeze_encoder=False):
    tok = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path).to(device)

    if freeze_encoder and hasattr(model, "model") and hasattr(model.model, "encoder"):
        for p in model.model.encoder.parameters():
            p.requires_grad = False
        if hasattr(model.model, "shared"):
            for p in model.model.shared.parameters():
                p.requires_grad = False

    def tokenize(batch):
        x = tok(batch["src"], truncation=True, max_length=256)
        y = tok(text_target=batch["tgt"], truncation=True, max_length=256)
        x["labels"] = y["input_ids"]
        return x

    train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
    dev_tok = dev_ds.map(tokenize, batched=True, remove_columns=dev_ds.column_names)

    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
    )

    collator = DataCollatorForSeq2Seq(tok, model=model)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=dev_tok, data_collator=collator)
    trainer.train()

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    return out_dir

In [ ]:
# Verify LoRA works with Helsinki NLP
def find_lora_linear_module_names(model, include_substring="decoder", endswith=("q_proj","k_proj","v_proj","out_proj")):
    """Return full module names of Linear layers in (e.g.) decoder that end with attention proj names."""
    names = []
    for name, module in model.named_modules():
        if include_substring in name and name.endswith(endswith) and isinstance(module, nn.Linear):
            names.append(name)
    return names

def load_lora_for_eval(base_model_name_or_path, adapter_dir):
    """Load BASE model + LoRA adapter for inference."""
    base = AutoModelForSeq2SeqLM.from_pretrained(base_model_name_or_path).to(device)
    model = PeftModel.from_pretrained(base, adapter_dir).to(device)
    tok = AutoTokenizer.from_pretrained(adapter_dir, use_fast=False)
    return model, tok

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType

def fine_tune_lora(
    base_model_name_or_path,
    train_ds,
    dev_ds,
    out_dir,
    lr,
    epochs,
    batch_size=8,
    freeze_encoder=True,
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    early_stopping_patience=None,
    weight_decay=0.0,
    lora_decoder_only=True,
    adapter_path=None,
    max_source_len=256,
    max_target_len=256,
):
    # Tokenizer + Base model ---
    tok = AutoTokenizer.from_pretrained(base_model_name_or_path, use_fast=False)
    base = AutoModelForSeq2SeqLM.from_pretrained(base_model_name_or_path).to(device)

    # Encoder Freezing option
    if freeze_encoder and hasattr(base, "model") and hasattr(base.model, "encoder"):
        for p in base.model.encoder.parameters():
            p.requires_grad = False
        if hasattr(base.model, "shared"):
            for p in base.model.shared.parameters():
                p.requires_grad = False

    # Attach / load LoRA
    if adapter_path is not None:
        # continue from existing adapter
        model = PeftModel.from_pretrained(base, adapter_path).to(device)
        model.train()
        try:
            model.set_adapter("default")
        except Exception:
            pass

        # Freeze everything and unfreeze only LoRA params (robust)
        for n, p in model.named_parameters():
            p.requires_grad = False
        for n, p in model.named_parameters():
            if "lora_" in n:
                p.requires_grad = True

    else:
        # create a new adapter
        if lora_decoder_only:
            target_modules = find_lora_linear_module_names(base, include_substring="decoder")
            if len(target_modules) == 0:
                raise ValueError("No decoder Linear target modules found for LoRA. Inspect model.named_modules().")
        else:
            target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"]

        lora_cfg = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=target_modules,
            bias="none",
        )
        model = get_peft_model(base, lora_cfg).to(device)

    model.print_trainable_parameters()

    # Tokenize datasets
    def tokenize(batch):
        x = tok(batch["src"], truncation=True, max_length=max_source_len)
        y = tok(text_target=batch["tgt"], truncation=True, max_length=max_target_len)
        x["labels"] = y["input_ids"]
        return x

    train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
    dev_tok   = dev_ds.map(tokenize, batched=True, remove_columns=dev_ds.column_names)

    collator = DataCollatorForSeq2Seq(tok, model=model)

    # Training args
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        weight_decay=weight_decay,
    )

    callbacks = []
    if early_stopping_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    # Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=collator,
        callbacks=callbacks,
    )

    trainer.train()

    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)

    return out_dir

### Run Idiom-only Fine-Tuning

In [ ]:
IDIOM_ONLY_DIR = os.path.join(CHECKPOINT_DIR, "idiom_only_v1")

# Set Fine-tuning Hyperparameters
lr = 5e-5
epochs = 3
batch_size = 8

# Run Fine-tuning
fine_tune(BASE_MODEL, idiom_ds["train"], idiom_ds["dev"], IDIOM_ONLY_DIR, lr=lr, epochs=epochs, batch_size=batch_size)

idiom_model = AutoModelForSeq2SeqLM.from_pretrained(IDIOM_ONLY_DIR).to(device)
idiom_tok = AutoTokenizer.from_pretrained(IDIOM_ONLY_DIR, use_fast=False)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,1.186557,1.116577
2,0.733086,1.105641
3,0.514618,1.111506


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## Stage 2: PEFT/LoRA

Decoding with ranks {4,8,16)

In [ ]:
LORA_RANKS = [4, 8, 16]

# LoRA parameters
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training hyperparameters
lr_stage1 = 5e-5
epochs_stage1 = 3
batch_size_stage1 = 8

lr_stage2 = 1e-5
epochs_stage2 = 5
batch_size_stage2 = 8
early_stop = 2

In [ ]:
def run_lora_two_stage_for_rank(r):
    stage1_dir = os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage1_idioms")
    stage2_dir = os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage2_wmt_encfrozen")

    # Stage 1: idioms
    fine_tune_lora(
        BASE_MODEL,
        idiom_ds["train"],
        idiom_ds["dev"],
        stage1_dir,
        lr=lr_stage1,
        epochs=epochs_stage1,
        batch_size=batch_size_stage1,
        freeze_encoder=False,
        lora_r=r,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        early_stopping_patience=None,
        weight_decay=0.0,
        lora_decoder_only=True,
        adapter_path=None,
    )

    # Stage 2: WMT (encoder frozen), continue adapter
    fine_tune_lora(
        BASE_MODEL,
        general_ds["ft_train"],
        general_ds["ft_dev"],
        stage2_dir,
        lr=lr_stage2,
        epochs=epochs_stage2,
        batch_size=batch_size_stage2,
        freeze_encoder=True,
        lora_r=r,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        early_stopping_patience=early_stop,
        weight_decay=0.0,
        lora_decoder_only=True,
        adapter_path=stage1_dir,
    )

    return stage1_dir, stage2_dir


rank_runs = {}
for r in LORA_RANKS:
    print(f"\n===== Running LoRA rank sweep: r={r} =====")
    s1, s2 = run_lora_two_stage_for_rank(r)
    rank_runs[r] = {"stage1": s1, "stage2": s2}


===== Running LoRA rank sweep: r=4 =====


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 196,608 || all params: 134,102,528 || trainable%: 0.1466


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss
1,1.366688,1.306988
2,1.357231,1.285771
3,1.329413,1.280383


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 786,432 || all params: 134,692,352 || trainable%: 0.5839


Epoch,Training Loss,Validation Loss
1,2.229060,2.011800
2,2.110816,1.998773
3,1.836391,1.992116
4,2.156171,1.988741
5,2.048355,1.987702


In [ ]:
# check if checkpoints exist
for r in [4, 8, 16]:
    stage2_dir = os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage2_wmt_encfrozen")
    print(f"\n=== r={r} ===")
    print("Exists:", os.path.isdir(stage2_dir))
    if os.path.isdir(stage2_dir):
        print("Files:", os.listdir(stage2_dir))


=== r=4 ===
Exists: True
Files: ['checkpoint-2500', 'checkpoint-5000', 'checkpoint-7500', 'checkpoint-10000', 'checkpoint-12500', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']

=== r=8 ===
Exists: True
Files: ['checkpoint-2500', 'checkpoint-5000', 'checkpoint-7500', 'checkpoint-10000', 'checkpoint-12500', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']

=== r=16 ===
Exists: True
Files: ['checkpoint-2500', 'checkpoint-5000', 'checkpoint-7500', 'checkpoint-10000', 'checkpoint-12500', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'vocab.json', 'source.spm', 'target.spm']


In [ ]:
# create rank runs from checkpoints
LORA_RANKS = [4, 8, 16]

rank_runs = {
    r: {
        "stage1": os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage1_idioms"),
        "stage2": os.path.join(CHECKPOINT_DIR, f"lora_r{r}_stage2_wmt_encfrozen"),
    }
    for r in LORA_RANKS
}

# sanity print
for r, paths in rank_runs.items():
    print(f"\n=== r={r} ===")
    print("stage1 exists:", os.path.isdir(paths["stage1"]))
    print("stage2 exists:", os.path.isdir(paths["stage2"]))


=== r=4 ===
stage1 exists: True
stage2 exists: True

=== r=8 ===
stage1 exists: True
stage2 exists: True

=== r=16 ===
stage1 exists: True
stage2 exists: True


In [ ]:
def resolve_adapter_dir(stage2_dir):
    # use adapter files in stage2_dir
    if os.path.exists(os.path.join(stage2_dir, "adapter_config.json")):
        return stage2_dir

    # Otherwise, search checkpoint subfolders and pick the latest that has adapter_config.json
    ckpts = [d for d in os.listdir(stage2_dir) if d.startswith("checkpoint-")]
    ckpts = sorted(ckpts, key=lambda x: int(x.split("-")[1]))
    for ck in reversed(ckpts):
        cand = os.path.join(stage2_dir, ck)
        if os.path.exists(os.path.join(cand, "adapter_config.json")):
            return cand

    raise FileNotFoundError(f"No adapter_config.json found in {stage2_dir} or its checkpoint-* subfolders.")

In [ ]:
# Decoding sweep
def make_beam_lp_setups(beams=(4, 8), lps=(0.8, 1.0, 1.2)):
    setups = {"greedy": {}}
    for b in beams:
        for lp in lps:
            name = f"beam{b}_lp{lp}"
            setups[name] = {"num_beams": b, "length_penalty": lp, "early_stopping": True}
    return setups

DECODING_SETUPS = make_beam_lp_setups(beams=(4, 8), lps=(0.8, 1.0, 1.2))

In [ ]:
def eval_and_log_decoding_sweep(model, tok, run_prefix, model_dir, lr_used, epochs_used, batch_size_used, notes_prefix=""):
    # Idioms
    for decode_name, gen_kwargs in DECODING_SETUPS.items():
        idiom_pred = generate_preds(model, tok, idiom_test["src"], batch_size=16, gen_kwargs=gen_kwargs)
        idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
        print(f"{run_prefix} idioms ({decode_name}):", idiom_m)

        log_metrics(
            f"{run_prefix}_{decode_name}",
            "idioms_test",
            idiom_m,
            model_name=model_dir,
            learning_rate=lr_used,
            epochs=epochs_used,
            batch_size=batch_size_used,
            max_source_len=MAX_SOURCE_LEN,
            max_new_tokens=MAX_NEW_TOKENS,
            freeze_encoder=True,
            train_size=len(general_ds["ft_train"]),
            dev_size=len(general_ds["ft_dev"]),
            seed=SEED,
            n_eval=len(idiom_test),
            notes=f"{notes_prefix} | decoding={decode_name} | {gen_kwargs}",
        )

    # WMT
    for decode_name, gen_kwargs in DECODING_SETUPS.items():
        wmt_pred = generate_preds(model, tok, wmt_test["src"], batch_size=16, gen_kwargs=gen_kwargs)
        wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
        print(f"{run_prefix} wmt ({decode_name}):", wmt_m)

        log_metrics(
            f"{run_prefix}_{decode_name}",
            "wmt_test",
            wmt_m,
            model_name=model_dir,
            learning_rate=lr_used,
            epochs=epochs_used,
            batch_size=batch_size_used,
            max_source_len=MAX_SOURCE_LEN,
            max_new_tokens=MAX_NEW_TOKENS,
            freeze_encoder=True,
            train_size=len(general_ds["ft_train"]),
            dev_size=len(general_ds["ft_dev"]),
            seed=SEED,
            n_eval=len(wmt_test),
            notes=f"{notes_prefix} | decoding={decode_name} | {gen_kwargs}",
        )

## Stage 2 Evaluation

In [ ]:
# Eval
for r, paths in rank_runs.items():
    stage2_dir = resolve_adapter_dir(paths["stage2"])
    print(f"\n===== Evaluating LoRA r={r} adapter dir: {stage2_dir} =====")

    lora_model, lora_tok = load_lora_for_eval(BASE_MODEL, stage2_dir)

    run_prefix = f"lora_r{r}_two_stage"
    notes_prefix = f"LoRA 2-stage | r={r} | stage2=WMT enc_frozen | stage1=idioms | lr2={lr_stage2}"

    eval_and_log_decoding_sweep(
        lora_model,
        lora_tok,
        run_prefix=run_prefix,
        model_dir=stage2_dir,
        lr_used=lr_stage2,
        epochs_used=epochs_stage2,
        batch_size_used=batch_size_stage2,
        notes_prefix=notes_prefix
    )


===== Evaluating LoRA r=4 adapter dir: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/lora_r4_stage2_wmt_encfrozen =====


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


lora_r4_two_stage idioms (greedy): {'bleu': 41.809157894702444, 'chrf': 61.681716737905326}
lora_r4_two_stage idioms (beam4_lp0.8): {'bleu': 40.76897952973159, 'chrf': 60.62100568893813}
lora_r4_two_stage idioms (beam4_lp1.0): {'bleu': 41.16847579264805, 'chrf': 61.0947228039622}
lora_r4_two_stage idioms (beam4_lp1.2): {'bleu': 41.699884481079565, 'chrf': 61.38712242835669}
lora_r4_two_stage idioms (beam8_lp0.8): {'bleu': 40.65932639501937, 'chrf': 60.6803093575713}
lora_r4_two_stage idioms (beam8_lp1.0): {'bleu': 40.94217668693307, 'chrf': 61.18789903536428}
lora_r4_two_stage idioms (beam8_lp1.2): {'bleu': 41.37415377298673, 'chrf': 61.560170206172145}
lora_r4_two_stage wmt (greedy): {'bleu': 27.467714167775704, 'chrf': 58.434767711527854}
lora_r4_two_stage wmt (beam4_lp0.8): {'bleu': 27.614264495055373, 'chrf': 58.41591137426786}
lora_r4_two_stage wmt (beam4_lp1.0): {'bleu': 27.53968567685827, 'chrf': 58.41711906479945}
lora_r4_two_stage wmt (beam4_lp1.2): {'bleu': 27.49511231466325,

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


lora_r8_two_stage idioms (greedy): {'bleu': 42.1787655293237, 'chrf': 61.760908613641895}
lora_r8_two_stage idioms (beam4_lp0.8): {'bleu': 40.82255089546092, 'chrf': 60.79652099136028}
lora_r8_two_stage idioms (beam4_lp1.0): {'bleu': 41.35400706936225, 'chrf': 61.23290268862818}
lora_r8_two_stage idioms (beam4_lp1.2): {'bleu': 41.40149955323913, 'chrf': 61.32406516381892}
lora_r8_two_stage idioms (beam8_lp0.8): {'bleu': 40.8912239909261, 'chrf': 60.852900401198525}
lora_r8_two_stage idioms (beam8_lp1.0): {'bleu': 41.30680023128676, 'chrf': 61.35294432021068}
lora_r8_two_stage idioms (beam8_lp1.2): {'bleu': 41.60098757274608, 'chrf': 61.6074472343421}
lora_r8_two_stage wmt (greedy): {'bleu': 27.4803387263887, 'chrf': 58.45979563899476}
lora_r8_two_stage wmt (beam4_lp0.8): {'bleu': 27.600402953549036, 'chrf': 58.436460387688726}
lora_r8_two_stage wmt (beam4_lp1.0): {'bleu': 27.51667847547613, 'chrf': 58.44003299046878}
lora_r8_two_stage wmt (beam4_lp1.2): {'bleu': 27.47578346508769, 'chr

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


lora_r16_two_stage idioms (greedy): {'bleu': 41.72424543678229, 'chrf': 61.449820290497506}
lora_r16_two_stage idioms (beam4_lp0.8): {'bleu': 40.43789882744096, 'chrf': 60.81102492449115}
lora_r16_two_stage idioms (beam4_lp1.0): {'bleu': 40.68579128911464, 'chrf': 60.942062674724575}
lora_r16_two_stage idioms (beam4_lp1.2): {'bleu': 41.09159687674348, 'chrf': 61.209496654476425}
lora_r16_two_stage idioms (beam8_lp0.8): {'bleu': 40.781720601964174, 'chrf': 60.814745950211815}
lora_r16_two_stage idioms (beam8_lp1.0): {'bleu': 41.06687801234486, 'chrf': 61.23041322962609}
lora_r16_two_stage idioms (beam8_lp1.2): {'bleu': 41.54031696829093, 'chrf': 61.61378570161629}
lora_r16_two_stage wmt (greedy): {'bleu': 27.596422476853355, 'chrf': 58.459373444561535}
lora_r16_two_stage wmt (beam4_lp0.8): {'bleu': 27.72410526208743, 'chrf': 58.42349905037095}
lora_r16_two_stage wmt (beam4_lp1.0): {'bleu': 27.664991558260745, 'chrf': 58.457662774758724}
lora_r16_two_stage wmt (beam4_lp1.2): {'bleu': 27.